# Регрессия CC50

Прогнозирование CC50 (концентрация, вызывающая цитотоксичность на 50%)


In [1]:
import numpy as np
import pickle
RANDOM_STATE = 42
ASSETS_DIR = 'assets'
try:
    import seaborn as sns
    sns.set_style('whitegrid')
except ImportError:
    pass


In [2]:
import os
os.makedirs(ASSETS_DIR, exist_ok=True)
# Загрузка очищенных данных и результатов моделей
with open('_data.pkl', 'rb') as f:
    data = pickle.load(f)
df = data['df']
feat_cols = data['feat_cols']
target_cols = data['target_cols']
with open('_all_results.pkl', 'rb') as f:
    results = pickle.load(f)
stats = results['stats']
reg_results = results['reg_results']
print(f'Данные: {df.shape}, Признаки: {len(feat_cols)}')


Данные: (1001, 195), Признаки: 192


In [3]:
target = 'CC50, mM'
y = df[target]
use_log = stats[target]['skew'] > 1.5
y_use = np.log1p(y) if use_log else y.copy()
print(f'Целевая: {target}')
print(f'Лог-преобразование: {use_log} (skew={stats[target]["skew"]:.2f})')
print(f'Исходные: mean={y.mean():.2f}, median={y.median():.2f}, skew={y.skew():.2f}')


Целевая: CC50, mM
Лог-преобразование: True (skew=1.97)
Исходные: mean=589.11, median=411.04, skew=1.97


In [4]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
X_all = df[feat_cols].values
scaler = StandardScaler()
Xs = scaler.fit_transform(X_all)


In [5]:
models = [
    ('DummyMean', DummyRegressor, {'strategy': 'mean'}),
    ('DummyMedian', DummyRegressor, {'strategy': 'median'}),
    ('Ridge', Ridge, {'alpha': 1.0}),
    ('Lasso', Lasso, {'alpha': 0.01, 'max_iter': 5000}),
    ('KNN', KNeighborsRegressor, {'n_neighbors': 5}),
    ('RF', RandomForestRegressor, {'n_estimators': 100, 'random_state': RANDOM_STATE}),
    ('HGB', HistGradientBoostingRegressor, {'max_iter': 200, 'max_depth': 6, 'random_state': RANDOM_STATE}),
]

print(f'Модели для {target}:')
for name, ModelClass, kwargs in models:
    m = ModelClass(**kwargs)
    m.fit(Xs, y_use)
    preds = m.predict(Xs)
    mae_cv = cross_val_score(m, Xs, y_use, cv=kf, scoring='neg_mean_absolute_error')
    r2_cv = cross_val_score(m, Xs, y_use, cv=kf, scoring='r2')
    rmse_cv = np.sqrt(-cross_val_score(m, Xs, y_use, cv=kf, scoring='neg_mean_squared_error'))
    print(f'  {name}: MAE={-np.mean(mae_cv):.3f}(+/-{np.std(mae_cv):.3f}), R2={np.mean(r2_cv):.4f}, RMSE={np.mean(rmse_cv):.3f}')


Модели для CC50, mM:
  DummyMean: MAE=1.297(+/-0.020), R2=-0.0026, RMSE=1.587
  DummyMedian: MAE=1.265(+/-0.026), R2=-0.0836, RMSE=1.650
  Ridge: MAE=1.003(+/-0.067), R2=0.0214, RMSE=1.539
  Lasso: MAE=0.958(+/-0.044), R2=0.3457, RMSE=1.280


D:\ВУЗ мага\CML_vo_PJ\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] Не удается найти указанный файл
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "D:\ВУЗ мага\CML_vo_PJ\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
  File "D:\ВУЗ мага\CML_vo_PJ\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "C:\Users\NewGC\AppData\Local\Programs\Python\Python314\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:


  KNN: MAE=0.894(+/-0.085), R2=0.2669, RMSE=1.345
  RF: MAE=0.856(+/-0.032), R2=0.4222, RMSE=1.203
  HGB: MAE=0.843(+/-0.028), R2=0.4049, RMSE=1.221


In [6]:
best = max(reg_results[target], key=lambda k: reg_results[target][k]['R2_cv'])
r = reg_results[target][best]
print(f'\nЛучшая модель CV для CC50, mM: {best}')
print(f'  R2 CV: {r["R2_cv"]:.4f} (+/-{r["R2_cv_std"]:.4f})')
print(f'  MAE CV: {r["MAE_cv"]:.3f} (+/-{r["MAE_cv_std"]:.3f})')
print(f'  RMSE CV: {r["RMSE_cv"]:.3f} (+/-{r["RMSE_cv_std"]:.3f})')



Лучшая модель CV для CC50, mM: RF
  R2 CV: 0.4222 (+/-0.0432)
  MAE CV: -0.856 (+/-0.032)
  RMSE CV: 1.203 (+/-0.046)
